# 🤖 Affiliate Video Bot v7 — 2026
**AI tạo video affiliate viral cho TikTok & Shopee — 10 ngành hàng**

| Bước | Cell | Mô tả | Tần suất |
|------|------|--------|----------|
| 1 | **Cell 0** | Điền GitHub repo + ngrok token | 1 lần |
| 2 | **Cell 1** | Cài deps + clone code | 1 lần / session mới |
| 3 | **Cell 2** | Mount Drive + fonts | 1 lần |
| 4 | **Cell 3** | Tải AI models (tuỳ chọn) | 1 lần |
| 5 | **Cell 4** | 🚀 Khởi động server | **Mỗi session** |
| 6 | **Cell 5** | Test pipeline (tuỳ chọn) | Khi cần debug |

> ⚠️ **Chọn Runtime → T4 GPU trước khi chạy**

## ⚙️ Cell 0 — Cấu Hình
> Chỉ cần điền **GitHub repo** và **ngrok token**. Render URL sẽ tự phát hiện từ `render.yaml`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ⚙️  CELL 0 — CẤU HÌNH
# ═══════════════════════════════════════════════════════════════

import subprocess, sys, json, re
import urllib.request

# ── 1. GitHub repo chứa code bot ────────────────────────────
GITHUB_REPO = "https://github.com/YOUR_USERNAME/affiliate-video-bot.git"
# ↑ Thay YOUR_USERNAME bằng tên GitHub của bạn

# ── 2. ngrok authtoken (ngrok.com → Dashboard → Authtoken) ──
NGROK_AUTH_TOKEN = "your_ngrok_authtoken_here"

# ── 3. Telegram User ID (nhắn @userinfobot để lấy) ──────────
YOUR_TELEGRAM_ID = 0

# ═══════════════════════════════════════════════════════════════
# AUTO-DETECT từ đây trở xuống — KHÔNG CẦN CHỈNH
# ═══════════════════════════════════════════════════════════════

def _detect_render_url_from_repo(repo_url: str) -> str:
    """Đọc render.yaml từ GitHub để lấy service name → build Render URL."""
    # Chuyển git URL → raw content URL
    # https://github.com/USER/REPO.git → raw.githubusercontent.com/USER/REPO/main/render.yaml
    m = re.match(r'https://github\.com/([^/]+)/([^/\.]+)', repo_url)
    if not m:
        return ""
    user, repo = m.group(1), m.group(2)
    
    for branch in ("main", "master"):
        raw_url = f"https://raw.githubusercontent.com/{user}/{repo}/{branch}/render.yaml"
        try:
            with urllib.request.urlopen(raw_url, timeout=8) as resp:
                content = resp.read().decode()
            # Parse service name từ render.yaml (không cần yaml lib)
            name_match = re.search(r'^\s*name:\s*([\w-]+)', content, re.MULTILINE)
            if name_match:
                svc_name = name_match.group(1).strip()
                return f"https://{svc_name}.onrender.com"
        except Exception:
            continue
    return ""

def _detect_render_url_from_apis(candidates: list) -> str:
    """Ping từng candidate URL để tìm Render service đang sống."""
    for url in candidates:
        if not url: continue
        try:
            with urllib.request.urlopen(f"{url}/ping", timeout=6) as r:
                data = json.loads(r.read())
                if data.get("version"):
                    return url
        except Exception:
            continue
    return ""

def _detect_bot_secret_from_render(render_url: str) -> str:
    """Lấy COLAB_SECRET từ /health endpoint nếu Render expose."""
    # Render /health không expose secret — dùng default
    return "affiliatebot_v7_secret"

# ── Auto-detect RENDER_URL ───────────────────────────────────
print("🔍 Đang tự phát hiện Render URL...")

RENDER_URL = ""
BOT_SECRET = "affiliatebot_v7_secret"

# Bước 1: Đọc render.yaml từ GitHub repo
if "YOUR_USERNAME" not in GITHUB_REPO:
    RENDER_URL = _detect_render_url_from_repo(GITHUB_REPO)
    if RENDER_URL:
        print(f"   ✅ Tìm thấy từ render.yaml: {RENDER_URL}")

# Bước 2: Fallback — thử các pattern phổ biến từ repo name
if not RENDER_URL:
    m = re.match(r'https://github\.com/([^/]+)/([^/\.]+)', GITHUB_REPO)
    if m:
        repo_slug = m.group(2).lower()
        candidates = [
            f"https://{repo_slug}.onrender.com",
            f"https://{repo_slug}-v7.onrender.com",
            f"https://affiliate-video-bot-v7.onrender.com",
        ]
        print(f"   🔄 Ping {len(candidates)} candidates...")
        RENDER_URL = _detect_render_url_from_apis(candidates)
        if RENDER_URL:
            print(f"   ✅ Render đang sống: {RENDER_URL}")

# Bước 3: Lấy BOT_SECRET từ Render nếu có
if RENDER_URL:
    try:
        with urllib.request.urlopen(f"{RENDER_URL}/health", timeout=6) as r:
            h = json.loads(r.read())
            # /health trả version + engine, không trả secret → dùng default
            print(f"   ✅ Render health: version={h.get('version')} engine={h.get('engine')}")
    except Exception as e:
        print(f"   ⚠️  Render ping fail: {e} (có thể đang sleep, sẽ wake sau)")

# ── Validation ───────────────────────────────────────────────
print()
print("=" * 55)
issues = []
if "YOUR_USERNAME" in GITHUB_REPO:     issues.append("⚠️  GITHUB_REPO chưa điền")
if "your_ngrok" in NGROK_AUTH_TOKEN:   issues.append("⚠️  NGROK_AUTH_TOKEN chưa điền")
if YOUR_TELEGRAM_ID == 0:              issues.append("⚠️  YOUR_TELEGRAM_ID chưa điền (nhắn @userinfobot)")
if not RENDER_URL:                      issues.append("⚠️  Không tìm thấy Render URL — điền thủ công bên dưới")

if issues:
    for i in issues: print(i)
    if not RENDER_URL:
        print()
        print("👉 Nếu auto-detect thất bại, bỏ comment dòng dưới:")
        print("   # RENDER_URL = 'https://your-service.onrender.com'")
else:
    print(f"✅ GitHub  : {GITHUB_REPO[:55]}")
    print(f"✅ Render  : {RENDER_URL}")
    print(f"✅ Secret  : {BOT_SECRET}")
    print(f"✅ Telegram: {YOUR_TELEGRAM_ID}")
    print()
    print("→ Chạy Cell 1 tiếp theo ✓")
print("=" * 55)

# Uncomment nếu auto-detect không tìm được:
# RENDER_URL = "https://your-service.onrender.com"


## 📦 Cell 1 — Cài Đặt & Clone Code
> Chạy **1 lần duy nhất** (~8-10 phút). Session mới chỉ cần chạy lại nếu `/content` bị xoá.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 📦 CELL 1 — CÀI DEPENDENCIES + CLONE CODE
# ═══════════════════════════════════════════════════════════════

import subprocess, sys, os

REPO_DIR = "/content/affiliate-video-bot"

# ── System packages ───────────────────────────────────────────
print("📦 1/4 ffmpeg...")
subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg", "libsm6", "libxext6"], check=True)
print("✅ ffmpeg OK")

# ── Python packages ───────────────────────────────────────────
print("\n📦 2/4 Python base packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "python-telegram-bot==21.6", "flask", "requests", "pyngrok",
    "Pillow", "moviepy==1.0.3", "imageio", "imageio-ffmpeg", "numpy", "tqdm",
], check=True)
print("✅ Base OK")

print("\n📦 3/4 AI/ML packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "diffusers>=0.30.0", "transformers>=4.44.0",
    "accelerate>=0.34.0", "safetensors>=0.4.4",
    "huggingface_hub>=0.24.6", "xformers",
], check=True)
print("✅ AI/ML OK")

print("\n📦 4/4 AudioCraft (nhạc AI)...")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "audiocraft"], check=True)
    print("✅ AudioCraft OK")
except Exception as e:
    print(f"⚠️  Skip AudioCraft: {e} → dùng Pixabay fallback")

# ── Clone / pull repo ─────────────────────────────────────────
print("\n📥 Clone code từ GitHub...")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)
    print("✅ Clone OK")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("✅ Pull latest OK")

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print("\n" + "="*50)
print("✅ CÀI ĐẶT HOÀN TẤT → Chạy Cell 2")
print("="*50)


## 🔗 Cell 2 — Mount Google Drive + Fonts
> Chạy **1 lần**. Fonts lưu vào Drive — session sau load lại trong <5 giây.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🔗 CELL 2 — MOUNT DRIVE + SETUP FONTS
# ═══════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, "/content/affiliate-video-bot")

from pipeline.drive_manager import setup_drive

print("🔗 Mount Google Drive...")
drive = setup_drive()

print("\n📊 Drive stats:")
for folder, info in drive.drive_stats().items():
    print(f"   📁 {folder}/: {info['size_mb']} MB | {info['files']} files")

print("\n🔤 Fonts (lần đầu ~30 giây)...")
for font in ["Montserrat-Bold.ttf", "Montserrat-ExtraBold.ttf", "BeVietnamPro-Bold.ttf"]:
    path = drive.get_font_path(font)
    print(f"   {'✅' if path else '⚠️ '} {font}")

print("\n" + "="*50)
print(f"✅ DRIVE: {drive.drive_root}")
print("→ Chạy Cell 3 (tuỳ chọn) hoặc thẳng Cell 4")
print("="*50)


## 🤖 Cell 3 — Tải AI Models (Tuỳ Chọn)
> Tải **1 lần** → lưu Drive. Session sau load từ Drive, không tải lại.
> **Bỏ qua** nếu chỉ dùng MoviePy engine (không cần GPU).

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🤖 CELL 3 — TẢI AI MODELS (uncomment model muốn dùng)
# ═══════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, "/content/affiliate-video-bot")
from pipeline.drive_manager import setup_drive
drive = setup_drive()

# Model 1: Wan2.1-I2V — Tốt nhất, cinematic, cần 12GB VRAM, ~25GB disk
# drive.download_model("Wan-AI/Wan2.1-I2V-14B-480P", "wan2.1-i2v-14B-480P")

# Model 2: CogVideoX-5B — Nhanh hơn, Text-to-Video, 8GB VRAM, ~18GB disk
# drive.download_model("THUDM/CogVideoX-5b", "cogvideox-5b")

# Model 3: FLUX.1-schnell — Ảnh AI chất lượng cao, 4 steps, ~8GB disk
# drive.download_model("black-forest-labs/FLUX.1-schnell", "flux1-schnell")

# Model 4: AnimateDiff — Nhẹ nhất, 6GB VRAM, ~7GB disk
# drive.download_model("guoyww/animatediff-motion-adapter-sdxl-beta", "animatediff-sdxl")

# Model 5: MusicGen-small — Nhạc AI, ~300MB
# drive.download_model("facebook/musicgen-small", "musicgen-small")

print("💡 Colab Free T4 = 15GB VRAM → Wan2.1 hoặc CogVideoX đều được")
print("   Không tải model → dùng MoviePy fallback (không cần GPU)")
print("→ Chạy Cell 4 để khởi động server")


## 🚀 Cell 4 — KHỞI ĐỘNG SERVER
> **Chạy mỗi session.** Tự động:
> - Mount Drive, start Flask server
> - Tạo ngrok tunnel
> - **Tự đăng ký URL về Render** (không cần copy-paste)
> - Gửi thông báo Telegram

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🚀 CELL 4 — KHỞI ĐỘNG SERVER (chạy mỗi session)
# ═══════════════════════════════════════════════════════════════

import sys, os, json, threading, time, traceback
import requests as req_lib
sys.path.insert(0, "/content/affiliate-video-bot")

REPO_DIR = "/content/affiliate-video-bot"
FLASK_PORT = 5001

# ── Mount Drive ───────────────────────────────────────────────
from pipeline.drive_manager import setup_drive
drive = setup_drive()

# ── GPU Info ─────────────────────────────────────────────────
def get_gpu_info():
    try:
        import torch
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            mem  = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
            return f"{name} ({mem}GB VRAM)"
        return "CPU only"
    except:
        return "GPU check failed"

# ── Pipeline runner ───────────────────────────────────────────
def run_pipeline(user_id, name, price, description, platform, callback_url):
    print(f"\n📥 Task: {name} | {price} | {platform}")
    try:
        from pipeline.product_analyzer import analyze_product
        from pipeline.emotional_engine  import build_emotional_package
        from pipeline.viral_caption     import generate_viral_caption
        from pipeline.background        import get_main_prompt, get_hook_prompt
        from pipeline.video_engine      import generate_video

        analysis  = analyze_product(name, description, price)
        emotional = build_emotional_package(name, analysis.category, analysis.gender, price)
        vc        = generate_viral_caption(name, price, analysis.category,
                                           analysis.gender, emotional, platform)
        caption   = vc.tiktok if platform == "tiktok" else vc.shopee

        output = generate_video(
            ai_prompt   = get_main_prompt(analysis.category, analysis.gender, product_name=name),
            hook_prompt = get_hook_prompt(analysis.category, analysis.gender),
            product_name= name, price=price, category=analysis.category,
            music_mood  = emotional.emotional_music,
            value_stack = "✅ Freeship toàn quốc\n✅ Đổi trả 7 ngày\n✅ Giao 2-3 ngày",
            cta         = vc.cta_bio,
            badge       = emotional.urgency_line[:40],
            gender      = analysis.gender,
            engine      = "auto",
        )

        if output and output.exists():
            size_mb = output.stat().st_size / 1e6
            print(f"✅ Video: {output} ({size_mb:.1f} MB)")
            send_callback(callback_url, {
                "user_id": user_id, "status": "success_drive",
                "video_url": str(output), "caption": caption[:900], "error": ""
            })
            from IPython.display import Video, display, Markdown
            display(Markdown(f"### ✅ {name} ({size_mb:.1f}MB)"))
            display(Video(str(output), width=360))
        else:
            raise RuntimeError("generate_video trả về None hoặc file không tồn tại")

    except Exception as e:
        traceback.print_exc()
        send_callback(callback_url, {
            "user_id": user_id, "status": "error",
            "error": str(e)[:200], "caption": "", "video_url": ""
        })

def send_callback(url, payload):
    if not url: return
    try:
        req_lib.post(url, json=payload,
                     headers={"X-Bot-Secret": BOT_SECRET}, timeout=15)
        print(f"✅ Callback → {url}")
    except Exception as e:
        print(f"⚠️  Callback fail: {e}")

# ── Flask App ─────────────────────────────────────────────────
from flask import Flask, request, jsonify
colab_app = Flask("affiliate_colab_v7")

@colab_app.route("/ping")
def ping():
    return jsonify({"status": "alive", "gpu": get_gpu_info(), "version": "7.0"}), 200

@colab_app.route("/info")
def info():
    return jsonify({"status": "alive", "gpu": get_gpu_info(),
                    "drive": drive.drive_stats(), "version": "7.0"}), 200

@colab_app.route("/generate", methods=["POST"])
def generate():
    data   = request.get_json(silent=True) or {}
    secret = data.get("secret", "")
    if secret != BOT_SECRET:
        return jsonify({"error": "Unauthorized"}), 403
    threading.Thread(
        target=run_pipeline,
        args=(data.get("user_id"), data.get("name", "SP"),
              data.get("price", ""), data.get("description", ""),
              data.get("platform", "tiktok"), data.get("callback_url", "")),
        daemon=True
    ).start()
    return jsonify({"status": "accepted"}), 200

# ── Start Flask ───────────────────────────────────────────────
threading.Thread(
    target=lambda: colab_app.run(host="0.0.0.0", port=FLASK_PORT, use_reloader=False),
    daemon=True
).start()
time.sleep(2)
print(f"✅ Flask đang chạy port {FLASK_PORT}")

# ── Ngrok tunnel ──────────────────────────────────────────────
from pyngrok import ngrok, conf as ngrok_conf

if NGROK_AUTH_TOKEN and "your_ngrok" not in NGROK_AUTH_TOKEN:
    ngrok_conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print("✅ ngrok authenticated")
else:
    print("⚠️  NGROK_AUTH_TOKEN chưa set — URL expire sau 2h")

try: ngrok.kill()
except: pass
time.sleep(1)

tunnel    = ngrok.connect(FLASK_PORT, bind_tls=True)
ngrok_url = tunnel.public_url
gpu_info  = get_gpu_info()

print()
print("=" * 60)
print("🚀 COLAB SERVER V7 ĐANG CHẠY!")
print("=" * 60)
print(f"🔗 ngrok URL  : {ngrok_url}")
print(f"🖥️  GPU        : {gpu_info}")
print(f"🌐 Render     : {RENDER_URL or '❌ chưa detect được'}")
print("=" * 60)

# ── Tự đăng ký về Render ─────────────────────────────────────
_registered = False
if RENDER_URL:
    print("\n📡 Tự đăng ký URL về Render...")
    try:
        resp = req_lib.post(
            f"{RENDER_URL}/colab/seturl",
            json={"url": ngrok_url, "notify_user_id": YOUR_TELEGRAM_ID},
            headers={"X-Bot-Secret": BOT_SECRET, "Content-Type": "application/json"},
            timeout=25
        )
        if resp.status_code == 200:
            _registered = True
            print("✅ Đăng ký thành công!")
            if YOUR_TELEGRAM_ID:
                print(f"📲 Đã gửi thông báo → Telegram {YOUR_TELEGRAM_ID}")
        else:
            print(f"⚠️  Render {resp.status_code} — {resp.text[:80]}")
    except Exception as e:
        print(f"⚠️  Tự đăng ký fail: {e}")
else:
    print("\n⚠️  Render URL chưa detect — chạy lại Cell 0 hoặc gửi thủ công:")

if not _registered:
    print(f"\n👉 Gửi lệnh này vào Telegram Bot:")
    print(f"   /setcolab {ngrok_url}")

print()
print("📌 GIỮ NOTEBOOK NÀY MỞ — đóng tab Colab ngủ!")
print("💡 /autocolab on để bot tự ping giữ Colab sống")


## 🧪 Cell 5 — Test Pipeline (Không cần GPU)
> Kiểm tra product analyzer + emotional engine + caption.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🧪 CELL 5 — TEST PIPELINE (tuỳ chọn)
# ═══════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, "/content/affiliate-video-bot")

from pipeline.product_analyzer import analyze_product
from pipeline.emotional_engine  import build_emotional_package
from pipeline.viral_caption     import generate_viral_caption

test_cases = [
    ("Serum Vitamin C 30ml",  "350k",  "Serum trắng da niacinamide 10%"),
    ("Váy maxi hoa nhí",      "299k",  "Váy nữ lụa mềm tay dài"),
    ("Collagen uống Nhật",    "890k",  "Collagen peptide 10000mg Shiseido"),
    ("Đèn decor phòng ngủ",   "185k",  "Đèn LED cảm ứng dán tường"),
    ("Thức ăn mèo Whiskas",   "65k",   "Pate mèo vị cá hồi 400g"),
    ("Bộ bodysuit sơ sinh",   "145k",  "Bodysuit cotton 100% bé 0-12 tháng"),
]

print("🧪 KIỂM TRA PIPELINE\n" + "="*60)
ok = err = 0
for name, price, desc in test_cases:
    try:
        a  = analyze_product(name, desc, price)
        ep = build_emotional_package(name, a.category, a.gender, price)
        vc = generate_viral_caption(name, price, a.category, a.gender, ep, "tiktok")
        print(f"✅ {name:<30} → {a.category} | {a.gender}")
        ok += 1
    except Exception as e:
        print(f"❌ {name:<30} → {e}")
        err += 1

print("="*60)
print(f"Kết quả: {ok} OK / {err} lỗi")
if not err:
    print("✅ Pipeline hoạt động tốt → Chạy Cell 4 để khởi động server")


## 🎬 Cell 6 — Tạo Video Thử (Test GPU)
> Kiểm tra toàn bộ AI pipeline từ đầu đến cuối.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🎬 CELL 6 — TẠO VIDEO THỬ
# ═══════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, "/content/affiliate-video-bot")

from pipeline.product_analyzer import analyze_product
from pipeline.emotional_engine  import build_emotional_package
from pipeline.viral_caption     import generate_viral_caption
from pipeline.background        import get_main_prompt, get_hook_prompt
from pipeline.video_engine      import generate_video
from IPython.display            import Video, display, Markdown

TEST_NAME  = "Serum Vitamin C 30ml"
TEST_PRICE = "350k"
TEST_DESC  = "Serum trắng da niacinamide 10% Hàn Quốc"
TEST_PLAT  = "tiktok"

print(f"🎬 Tạo video: {TEST_NAME}\n" + "="*50)

analysis  = analyze_product(TEST_NAME, TEST_DESC, TEST_PRICE)
emotional = build_emotional_package(TEST_NAME, analysis.category, analysis.gender, TEST_PRICE)
vc        = generate_viral_caption(TEST_NAME, TEST_PRICE, analysis.category,
                                    analysis.gender, emotional, TEST_PLAT)

print(f"  Ngành  : {analysis.category}")
print(f"  Target : {analysis.gender}")
print(f"  Nhạc   : {emotional.emotional_music}")
print(f"  Hook   : {emotional.hook_curiosity[:60]}")

output = generate_video(
    ai_prompt   = get_main_prompt(analysis.category, analysis.gender, product_name=TEST_NAME),
    hook_prompt = get_hook_prompt(analysis.category, analysis.gender),
    product_name= TEST_NAME, price=TEST_PRICE, category=analysis.category,
    music_mood  = emotional.emotional_music,
    value_stack = "✅ Freeship\n✅ Đổi trả 7 ngày\n✅ Giao 2-3 ngày",
    cta         = vc.cta_bio, badge=emotional.urgency_line[:40],
    gender      = analysis.gender, engine="auto",
)

if output and output.exists():
    size_mb = output.stat().st_size / 1e6
    display(Markdown(f"### ✅ Video OK ({size_mb:.1f} MB)"))
    display(Video(str(output), width=360))
    print(f"\n📋 Caption TikTok:\n{vc.tiktok[:400]}")
    print(f"\n🔬 A/B Hooks:")
    for i, h in enumerate(getattr(vc, 'hook_ab_variants', emotional.ab_hooks)[:3], 1):
        print(f"  Hook {i}: {h}")
else:
    print("❌ Video generation thất bại — kiểm tra GPU và model")
